# Figure 2 — duplicate standard syntheses (S45)

Time-resolved extinction spectra of two replicate samples (`S45-M01`, sample_1 and sample_3; 2.8 µL/s × 5, Q_tot = 14 µL/s),
AvaSpec-ULS4096CL-EVO spectrometer (3198 px). Final spectrum = mean of the last 5 smoothed spectra; λmax and λ3 = parabolic vertex,
λ2 = shoulder (2nd-derivative minimum) on the mean of the two final spectra. Code: `fig02.py`.
The TEM insert is added in the final layout.

## Analysis code
All the code of this figure is in this notebook.

### `paths`

In [ ]:
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


### `io`

In [ ]:
"""Reading the instrument HDF5 files (both schemas)."""
from dataclasses import dataclass

import h5py
import numpy as np


@dataclass
class Sample:
    sample_id: str
    wavelengths: np.ndarray      # (n_wl,)
    absorbances: np.ndarray      # (n_spectra, n_wl), time-ordered
    flows_set: np.ndarray        # (5,) commanded flows, µL/s


def load_measurement_csv(path):
    """Instrument measurement CSV (`;`, header = timestamp + wavelengths, one spectrum per row).

    Returns (wavelengths, elapsed seconds since the first spectrum, absorbances (n_spectra, n_wl)).
    """
    import pandas as pd

    df = pd.read_csv(path, header=None, sep=";")
    wl = df.iloc[0, 1:].astype(float).to_numpy()
    times = pd.to_datetime(df.iloc[1:, 0])
    t_sec = (times - times.iloc[0]).dt.total_seconds().to_numpy()
    spectra = df.iloc[1:, 1:].apply(pd.to_numeric, errors="coerce").to_numpy()
    return wl, t_sec, spectra


def load_sample(path):
    """Load one sample H5.

    Format 2fast-sample 2.0. Returns None for a file without commanded flows.
    """
    with h5py.File(path, "r") as f:
        if "conditions/flow_set" not in f:
            return None
        flows = f["conditions/flow_set"][()]
        wl = f["spectra/wavelength"][()]
        absorb = f["spectra/extinction"][()]
        sample_id = str(f["sample"].attrs.get("sample_id", path.stem))
    absorb = np.asarray(absorb, dtype=float)
    if absorb.ndim == 1:
        absorb = absorb.reshape(1, -1)
    return Sample(sample_id, np.asarray(wl, dtype=float), absorb, np.asarray(flows, dtype=float))


### `peaks`

In [ ]:
"""Peak and shoulder positions (definitions used for Fig 2, authoritative per decision 2026-10-09)."""
import numpy as np
from scipy.signal import savgol_filter


def parabolic_peak(wl, spectrum, window, half_width_nm):
    """Vertex of a parabola fitted ±half_width_nm around the argmax inside `window`.
    Falls back to the argmax if the fit is not concave or the vertex leaves the fit range."""
    mask = (wl >= window[0]) & (wl <= window[1])
    idx = np.where(mask)[0][np.argmax(spectrum[mask])]
    half_w = int(round(half_width_nm / (wl[1] - wl[0])))
    lo, hi = max(0, idx - half_w), min(len(spectrum), idx + half_w + 1)
    x, y = wl[lo:hi], spectrum[lo:hi]
    a, b, c = np.polyfit(x, y, 2)
    if a >= 0:
        return wl[idx], spectrum[idx]
    vertex = -b / (2 * a)
    if not (x[0] <= vertex <= x[-1]):
        return wl[idx], spectrum[idx]
    return vertex, np.polyval((a, b, c), vertex)


def shoulder(wl, spectrum, window, savgol_window):
    """Shoulder = minimum of the Savitzky–Golay 2nd derivative inside `window`."""
    d2 = savgol_filter(spectrum, window_length=savgol_window, polyorder=3, deriv=2, delta=wl[1] - wl[0])
    mask = (wl >= window[0]) & (wl <= window[1])
    idx = np.where(mask)[0][np.argmin(d2[mask])]
    return wl[idx], spectrum[idx]


### `preprocess`

In [ ]:
"""Spectrum preprocessing, identical to the analysis code used for the paper
(paperAnalysis/analyse_reproducibility/*.py). Parameters are scientific choices: do not change them.
"""
import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

ULS4096_PIXELS = 3198    # AvaSpec-ULS4096CL-EVO grid, 315–750 nm
USB2048_PIXELS = 1219    # AvaSpec-2048-USB2-RM grid, 310–1000 nm

# dead-pixel blocks of the ULS4096 (start, stop), filled with the value of the pixel just before
ULS4096_DEAD_PIXELS = ((1200, 1210), (1900, 1910), (2460, 2472))


def patch_dead_pixels(spectra):
    """ULS4096 dead-pixel blocks set to the value of the pixel just before (works on 1-D or (n_spectra, n_wl))."""
    s = np.array(spectra, dtype=float, copy=True)
    if s.shape[-1] == ULS4096_PIXELS:
        for start, stop in ULS4096_DEAD_PIXELS:
            s[..., start:stop] = s[..., start - 1:start]
    return s


def smooth(spectrum):
    """Spectrometer-specific dead-pixel patch + Savitzky–Golay smoothing."""
    s = patch_dead_pixels(spectrum)
    n = len(s)
    if n == ULS4096_PIXELS:
        return savgol_filter(s, window_length=101, polyorder=3)
    if n == USB2048_PIXELS:
        return savgol_filter(s, window_length=51, polyorder=3)
    w = min(51, n // 10)
    if w % 2 == 0:
        w += 1
    return savgol_filter(s, window_length=w, polyorder=3)


def smooth_all(absorbances):
    return np.array([smooth(row) for row in absorbances])


def final_spectrum(smoothed, n_tail):
    """'Final' spectrum = mean of the last `n_tail` smoothed spectra of the measurement."""
    n_tail = min(n_tail, smoothed.shape[0])
    return smoothed[-n_tail:].mean(axis=0)


def harmonize(wavelengths, spectrum, wl_min, wl_max, n_points):
    """Linear interpolation onto a common grid; zero outside the measured range."""
    target = np.linspace(wl_min, wl_max, n_points)
    mask = (wavelengths >= wl_min) & (wavelengths <= wl_max)
    f = interp1d(wavelengths[mask], spectrum[mask], kind="linear", bounds_error=False, fill_value=0.0)
    return target, f(target)


### `fig02`

In [ ]:
"""Figure 2 — time-resolved spectra of duplicate standard syntheses (S45, ULS4096 spectrometer).

Port of paperAnalysis/analyse_reproducibility/plot_sample_overlay_S45.py (same data selection,
processing and peak definitions). The TEM insert is added in the final layout, not here.
"""
import numpy as np


PARAMS = dict(
    manip_dir=("S45", "S45-M01"),
    samples=("sample_1", "sample_3"),       # plotted as "Sample 1", "Sample 2"
    wl_min=315, wl_max=750, n_points=500,  # common grid
    n_tail=5,                               # spectra averaged into the final spectrum
    lambda3_window=(315, 360),
    shoulder_window=(370, 480),
    lambda_max_window=(450, 750),
    peak_fit_halfwidth_nm=40,               # λ3 uses a third of it, as in the original script
    shoulder_savgol_window=25,
)


def compute(params=PARAMS, raw=None):
    """Load, process and locate λmax / λ2 (shoulder) / λ3 on the mean of the selected samples."""
    raw = RAW if raw is None else raw
    manip = raw.joinpath(*params["manip_dir"])
    loaded = {}
    for h5 in sorted((manip / "samples").glob("*_sample_*.h5")):
        s = load_sample(h5)
        if s is not None:
            loaded[s.sample_id] = s

    samples = []
    for sid in params["samples"]:
        s = loaded[sid]
        smoothed = smooth_all(s.absorbances)
        final = final_spectrum(smoothed, params["n_tail"])
        grid = (params["wl_min"], params["wl_max"], params["n_points"])
        wl, spec = harmonize(s.wavelengths, final, *grid)
        kin = np.array([harmonize(s.wavelengths, row, *grid)[1] for row in smoothed])
        samples.append(dict(sample_id=sid, wl=wl, spectrum=spec, kinetics=kin))

    wl = samples[0]["wl"]
    mean = np.mean([s["spectrum"] for s in samples], axis=0)
    hw = params["peak_fit_halfwidth_nm"]
    lam3 = parabolic_peak(wl, mean, params["lambda3_window"], hw / 3)
    lam2 = shoulder(wl, mean, params["shoulder_window"], params["shoulder_savgol_window"])
    lmax = parabolic_peak(wl, mean, params["lambda_max_window"], hw)
    return dict(samples=samples, wl=wl, mean=mean,
                lambda_max=lmax, lambda_2=lam2, lambda_3=lam3)


STYLE = dict(fig_size=(6.0, 6.0), font_size=14, label_size=16, legend_size=12, tick_size=13,
             annotation_size=13, line_width=1.5, colors=("black", "#1f77b4"),
             kinetics_alpha=0.15, kinetics_linewidth=0.8, annotation_dy=0.05, arrow_shrink=2)


def plot(result, style=STYLE):
    import matplotlib.pyplot as plt

    rc = {"font.family": "sans-serif", "font.size": style["font_size"], "axes.labelsize": style["label_size"],
          "legend.fontsize": style["legend_size"], "xtick.labelsize": style["tick_size"],
          "ytick.labelsize": style["tick_size"], "axes.linewidth": 1.1, "xtick.direction": "in",
          "ytick.direction": "in", "xtick.top": True, "ytick.right": True}
    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=style["fig_size"])
        for i, s in enumerate(result["samples"]):
            color = style["colors"][i]
            for row in s["kinetics"]:
                ax.plot(s["wl"], row, color=color, lw=style["kinetics_linewidth"], alpha=style["kinetics_alpha"], zorder=1)
            ax.plot(s["wl"], s["spectrum"], color=color, lw=style["line_width"], zorder=3, label=f"Sample {i + 1}")
        for label, key in ((r"$\lambda_3$", "lambda_3"), (r"$\lambda_2$", "lambda_2"), (r"$\lambda_{max}$", "lambda_max")):
            x, y = result[key]
            ax.annotate(label, xy=(x, y), xytext=(x, y + style["annotation_dy"]), ha="center", va="bottom",
                        fontsize=style["annotation_size"],
                        arrowprops=dict(arrowstyle="->", shrinkA=0, shrinkB=style["arrow_shrink"], color="black"))
        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Extinction (a.u.)")
        ax.set_xlim(result["wl"][0], result["wl"][-1])
        ax.set_ylim(bottom=0)
        ax.legend(loc="upper left", frameon=False)
        fig.tight_layout()
    return fig


## Figure

In [ ]:
result = compute()
for key in ('lambda_max', 'lambda_2', 'lambda_3'):
    print(f"{key:10s} computed {result[key][0]:7.1f} nm")


In [ ]:
fig = plot(result)
FIGURES.mkdir(exist_ok=True)
for ext in ('png', 'pdf'):
    fig.savefig(FIGURES / f'fig02_standard_duplicates.{ext}', dpi=300, bbox_inches='tight', facecolor='white')
